In [72]:
from pymongo import MongoClient
import re
import os
from typing import Dict, Any, Union, List
from pymongo import MongoClient
import json
from dotenv import load_dotenv
# Langchain load data
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader
### LLMs - Langchain
import openai
from langchain_core.prompts import ChatPromptTemplate
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
### Summary chain
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.chains import LLMChain, MapReduceChain, load_summarize_chain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain_community.document_loaders import TextLoader

In [73]:
def connect_mongodb(uri,db_name, collection_name):
    # Connect to MongoDB 
    client = MongoClient(uri)
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]
    
    return collection
def extract_compliance_info(document: Dict[str, Any]) -> Union[Dict[str, Any], None]:
    """
    Extract _id, 3rd_integration, and privacy-compliance.violate if privacy-compliance.privacy-compliance is 'no'.

    Args:
        document (Dict[str, Any]): A single document from the MongoDB collection.

    Returns:
        Dict[str, Any] or None: A dictionary containing the extracted information if conditions are met, otherwise None.
    """
    if document.get("privacy-compliance", {}).get("privacy-compliance") == "no":
        return {
            "_id": document.get("_id"),
            "3rd_integration": document.get("3rd_integration"),
            "privacy-compliance.violate": document.get("privacy-compliance", {}).get("violate")
        }
    return None


def fetch_compliance_violations(collection) -> List[Dict[str, Any]]:
    """
    Fetch all documents from the collection and filter those with privacy compliance violations.
    
    Args:
        collection (pymongo.collection.Collection): The MongoDB collection to query.

    Returns:
        List[Dict[str, Any]]: A list of filtered documents with compliance violations.
    """
    results = []
    for document in collection.find():
        filtered_info = extract_compliance_info(document)
        if filtered_info:
            results.append(filtered_info)
    return results


def create_prompt_with_third_party_integration(third_party_integration: dict) -> str:
    """
    Generate a LangChain prompt for extracting import statements from third-party integrations.

    Args:
        third_party_integration (dict): A dictionary containing third-party integration details.

    Returns:
        str: A formatted prompt ready to be sent to a LangChain model.
    """
    # Convert the dictionary to a pretty-printed JSON string for the prompt
    third_party_integration_str = json.dumps(third_party_integration, indent=4)
    
    prompt_template = PromptTemplate(
        input_variables=["third_party_integration"],
        template="""
        Given that the app connects with the following third-party services:
        {third_party_integration}
        
        Provide me with the keywords or import statements that the app would use to connect with each third-party service above.

        Format the response as a JSON object where each third-party name is a key and the corresponding list of import statements is the value.

        Example format:
        {{
            "Google": ["import com.google.firebase.messaging.FirebaseMessagingService", "import com.google.firebase.messaging.RemoteMessage"],
            "Facebook": ["import com.facebook.FacebookSdk", "import com.facebook.appevents.AppEventsLogger"]
        }}
        If you can't find keywords or import statements then return as follows
        {{
            "keyword": []
        }}
        Provide the complete list of imports for each of the specified services.
        Please return the JSON object only and don't add any explanation or additional information.
        """
    )
    
    return prompt_template.format(third_party_integration=third_party_integration_str)
def extract_output_text_as_dict(response):
    """
    Extracts the 'output_text' from the response, cleans it, and converts it into a dictionary.

    :param response: The response dictionary containing 'output_text'.
    :return: A dictionary parsed from the 'output_text', or an empty dict if parsing fails.
    """
    if not response or not hasattr(response, "content"):  # Ensure response is valid
        print("Error: Invalid response object.")
        return {}

    output_text = response.content  # Assuming response.content holds the JSON string

    # Remove code block markers (```json and ```) if present
    if output_text.startswith("```json"):
        output_text = output_text[len("```json"):].strip("`").strip()

    try:
        # Convert the cleaned JSON string into a dictionary
        output_dict = json.loads(output_text)
        return output_dict
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return {}  # Return an empty dictionary instead of None
def insert_third_party_data(collection, condition_value: str, data: Dict[str, list]):
    """
    Insert or update the given dictionary data into MongoDB under the field 'keyword' using _id as condition.

    Args:
        collection (pymongo.collection.Collection): The MongoDB collection to interact with.
        condition_value (str): The _id value to use as a condition for insertion.
        data (Dict[str, list]): The dictionary data to insert or update.

    Returns:
        pymongo.results.UpdateResult: The result of the MongoDB update operation.
    """
    filter_query = {"_id": condition_value}
    update_query = {"$set": {"third-party-keyword": data}}
    
    # Insert or update the document using upsert=True
    result = collection.update_one(filter_query, update_query, upsert=True)
    return result

In [74]:
# server
mongoDB_uri = 'mongodb://192.168.1.14:27017'
mongoDB_database = 'wearable-project' 
mongoDB_collection = 'wearable-app'
# local
# mongoDB_uri = 'mongodb://localhost:27017'
# mongoDB_database = 'wearable-project' 
# mongoDB_collection = 'wearable-standalone-31-03-2025'
# root_directory = r'E:\wearable-standalone-code-extraction'
source_code_root_directory = r"E:\wearable-project-decompile-source\source-code-standalone"

In [75]:
#. Chat model
# Setup model
# Load environment variables
load_dotenv()

# Retrieve API key
api_key = os.getenv("OPENAI_API_KEY")
# Ensure the API key is correctly set
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables")

# Initialize the ChatOpenAI model
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    openai_api_key=api_key  # Ensure you explicitly pass the API key
)

In [76]:
# Connect to the MongoDB collection
collection = connect_mongodb(mongoDB_uri,mongoDB_database,mongoDB_collection)
# Fetch compliance violations
violations = fetch_compliance_violations(collection)

# Display results
for i in range(len(violations)):
    print("------------------ Loop-"+str(i)+" ------------------")
    data = violations[i]
    app_id = data["_id"]
    app_3rd_integration = data["3rd_integration"]
    # print(type(app_3rd_integration))
    print("App id: ", app_id)
    print("App 3rd party: ",app_3rd_integration)
    prompt_template = create_prompt_with_third_party_integration(app_3rd_integration)
    # print("prompt template: \n", prompt_template)
    response = llm.invoke(prompt_template)
    # print(response)
    output_dict = extract_output_text_as_dict(response)
    print("============= Output =============")
    print(output_dict)
    # print(type(output_dict))
    insert_status = insert_third_party_data(collection,app_id,output_dict)
    print("insert_status: ", insert_status)

------------------ Loop-414 ------------------
App id:  com.jp.fitness.biceps.exrcise
App 3rd party:  {'Google Services': 'com.google.android.c2dm.permission.RECEIVE, com.google.firebase.provider.FirebaseInitProvider, com.google.android.gms.auth.api.signin.RevocationBoundService, com.google.android.gms.permission.AD_ID', 'Flurry': 'com.flurry.android.agent.FlurryContentProvider'}
============= Output =============
{'Google Services': ['import com.google.android.c2dm.permission.RECEIVE', 'import com.google.firebase.provider.FirebaseInitProvider', 'import com.google.android.gms.auth.api.signin.RevocationBoundService', 'import com.google.android.gms.permission.AD_ID', 'import com.google.firebase.messaging.FirebaseMessagingService', 'import com.google.firebase.messaging.RemoteMessage', 'import com.google.android.gms.auth.api.signin.GoogleSignIn', 'import com.google.android.gms.auth.api.signin.GoogleSignInAccount', 'import com.google.android.gms.auth.api.signin.GoogleSignInClient', 'import 